# SOFTWARE ENGINEERING FOUNDATIONS FOR CLAUDE INTEGRATIONS

<br>

**About:** Core software engineering practices as applied to Claude API code: REST/JSON handling, sync vs. async patterns, error taxonomy, refactoring, version control, and code review discipline.

**Learning Goals:**
- Recognize the REST + JSON contract underlying the Anthropic Python SDK.
- Choose sync vs. async patterns for common Claude workloads (streaming, batches, agent loops).
- Distinguish transient (retry) from terminal (fail-fast) API error classes.
- Refactor a working prototype into a testable module with contracts.
- Review code for the anti-patterns most common in first-week Claude integrations.

**Keywords:** rest, json, async, error handling, refactoring, code review, contracts

**Prerequisite Knowledge:** (1) Python 3.11+, (2) HTTP/REST basics, (3) Lesson 01

**Target User:** Developer building a real integration, not a toy


**Estimated Cost to Run:** $0.02

**Estimated Completion:** 50-70 min

<hr style="border: 4px solid#003262;" />



#### CONTENTS

- [Part 1: SDK-over-REST - What the SDK Hides](#part-1-sdk-over-rest-what-the-sdk-hides)
- [Part 2: Sync vs. Async - When Each Fits](#part-2-sync-vs-async-when-each-fits)
- [Part 3: Error Handling - Transient vs. Terminal](#part-3-error-handling-transient-vs-terminal)
- [Part 4: Refactoring a Working Script into a Testable Module](#part-4-refactoring-a-working-script-becomes-a-testable-module)
- [Part 4b: Version Control for Claude API Code](#part-4b-version-control-for-claude-api-code)
- [Part 4c: Pytest Basics for Testing Claude Integrations](#part-4c-pytest-basics-for-testing-claude-integrations)
- [Part 5: Code Review Checklist for Claude Integrations](#part-5-code-review-checklist-for-claude-integrations)

<br>

In [1]:
# Paths in this repo are written relative to the repository root, but a
# Jupyter kernel starts in the notebook's own directory. Anchor to the root.
from ccdvf.paths import use_repo_root
use_repo_root()

# Setup - imports and environment check
import os
import sys
import json
import asyncio
import subprocess

import anthropic  # Verified against pypi.org/pypi/anthropic/json on 2026-09-02

from ccdvf.api import MessagesClient, CallResult
from ccdvf.api.errors import BudgetExceededError
from ccdvf.api.retry import with_retry
from ccdvf.config.model_registry import get_model
from ccdvf.security.secrets import load_api_key

DRY_RUN = os.getenv("CCDVF_DRY_RUN", "0") == "1"
if DRY_RUN:
    print("DRY RUN: live API calls will be skipped")
else:
    _ = load_api_key("ANTHROPIC_API_KEY")

haiku = get_model("haiku_current")

<hr style="border: 2px solid#003262;" />

## PART 1: SDK-over-REST - **What the SDK Hides**

Every call to `client.messages.create(...)` is a POST request to `https://api.anthropic.com/v1/messages` with a JSON body. The SDK handles authentication, serialization, deserialization, and error mapping. Understanding the raw HTTP shape helps you debug 400 errors and use raw `httpx` when the SDK does not fit.

Required headers (Verified against platform.claude.com/docs/en/api/getting-started on 2026-09-02):
- `x-api-key: <key>`
- `anthropic-version: 2023-06-01`
- `content-type: application/json`

Required body fields: `model`, `max_tokens`, `messages` (list with at least one entry).

In [ ]:
# Tier 1 - construct and display the raw request body shape
request_body = {
    "model": haiku.id,  # Verified against platform.claude.com/docs/en/docs/about-claude/models on 2026-09-02
    "max_tokens": 32,
    "messages": [{"role": "user", "content": "Reply with the word hello only."}],
}

print("POST https://api.anthropic.com/v1/messages")
print("Headers: x-api-key, anthropic-version: 2023-06-01, content-type: application/json")
print("Body:", json.dumps(request_body, indent=2))

if not DRY_RUN:
    raw_client = anthropic.Anthropic(api_key=load_api_key())
    resp = raw_client.messages.create(**request_body)
    print("\nResponse stop_reason:", resp.stop_reason)
    print("Response content[0].text:", resp.content[0].text)
else:
    print("DRY RUN: skipping live call (Part 1 Tier 1)")

POST https://api.anthropic.com/v1/messages
Headers: x-api-key, anthropic-version: 2023-06-01, content-type: application/json
Body: {
  "model": "claude-haiku-4-5-20251001",
  "max_tokens": 32,
  "messages": [
    {
      "role": "user",
      "content": "Reply with the word hello only."
    }
  ]
}

Response stop_reason: end_turn
Response content[0].text: hello


The SDK lets you add custom headers via `extra_headers` - useful for passing trace IDs through to Anthropic for request correlation.

In [3]:
# Tier 2 - custom headers via extra_headers for request tracing
if not DRY_RUN:
    raw_client = anthropic.Anthropic(api_key=load_api_key())
    resp = raw_client.messages.create(
        model=haiku.id,
        max_tokens=32,
        messages=[{"role": "user", "content": "Reply with the word hello only."}],
        extra_headers={"x-request-id": "notebook-02-demo"},
    )
    print("Response:", resp.content[0].text)
else:
    print("DRY RUN: skipping (Part 1 Tier 2)")

Response: hello


In [4]:
# Tier 3 - MessagesClient: production abstraction over the same call
if not DRY_RUN:
    with MessagesClient(model="haiku_current", budget_usd=0.01) as mc:
        result = mc.call(
            messages=[{"role": "user", "content": "Reply with the word hello only."}],
            max_tokens=32,
        )
    print("text:", result.text, "| cost_usd:", f"${result.cost_usd:.6f}")
else:
    print("DRY RUN: MessagesClient wraps POST /v1/messages with retry + cost tracking")

text: hello | cost_usd: $0.000034


<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **You send a Messages API request and receive a 400 response with `error.message: "messages: at least one message is required"`.** Which of the following request body fields is most likely missing or malformed?

(a) The `model` field is missing.

(b) The `messages` field is an empty list `[]`.

(c) The `max_tokens` field is 0.

(d) The `anthropic-version` header is missing.

<br>

In [ ]:
# CONCEPT CHECK: 400 error root cause
### YOUR CODE HERE ###
answer = "..."  # write 'a', 'b', 'c', or 'd' and one sentence justification

<hr style="border: 2px solid#003262;" />

<!-- ANSWER (for reviewer): (b) is correct. The error message names "messages: at least one message is required" - an empty list satisfies the type constraint but not the length constraint. (a) is wrong: missing model produces a different error. (c) is wrong: max_tokens 0 produces a separate validation error about range. (d) is wrong: a missing header check happens before body parsing and produces a different message. -->

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<hr style="border: 2px solid#003262;" />

## PART 2: **Sync** vs. **Async** - When Each Fits

The Anthropic Python SDK is synchronous by default. `AsyncAnthropic` provides the same interface for async code. 

When does async matter for Claude workloads?

- **Many parallel requests**: 50 independent I/O-bound API calls benefit from `asyncio.gather()` or a thread pool.

- **Streaming to end users via async web server**: use `AsyncAnthropic`; streaming itself is not async.

- **Batches API polling**: polling once every 30 seconds does NOT benefit from async.

The `ccdvf` modules are sync throughout - the right choice for Jupyter and batch scripts.

In [6]:
# Tier 1a - sync call (baseline)
if not DRY_RUN:
    sync_client = anthropic.Anthropic(api_key=load_api_key())
    resp = sync_client.messages.create(
        model=haiku.id, max_tokens=16,
        messages=[{"role": "user", "content": "Say sync."}],
    )
    print("Sync:", resp.content[0].text)
else:
    print("DRY RUN: skipping sync call")

Sync: sync


In [8]:
# Tier 1b - async call with AsyncAnthropic + asyncio.run
# Verified against pypi.org/pypi/anthropic/json on 2026-09-02
async def _async_call(api_key: str) -> str:
    async with anthropic.AsyncAnthropic(api_key=api_key) as ac:
        resp = await ac.messages.create(
            model=haiku.id, max_tokens=16,
            messages=[{"role": "user", "content": "Say async."}],
        )
    return resp.content[0].text

if not DRY_RUN:
    text = await _async_call (load_api_key())
    #text = asyncio.run(_async_call(load_api_key())) # use outside of ipynb
    print("Async:", text)
else:
    print("DRY RUN: skipping async call")

Async: async.


In [9]:
# Tier 2 - ccdvf.api clients are sync by design; document the tradeoff
print("ccdvf.api clients: sync only.")
print("For async use cases (FastAPI SSE, asyncio.gather over 100+ calls): use AsyncAnthropic directly.")
print("The ccdvf modules do not wrap AsyncAnthropic - async is a server concern, not a study concern.")

ccdvf.api clients: sync only.
For async use cases (FastAPI SSE, asyncio.gather over 100+ calls): use AsyncAnthropic directly.
The ccdvf modules do not wrap AsyncAnthropic - async is a server concern, not a study concern.


<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **You are building a batch summarizer that submits 500 requests to Anthropic overnight.** Which of the following is a genuine case for asyncio?

(a) Polling one Batches API job to completion with 30-second intervals.


(b) Submitting the 500 requests to the Messages API in parallel.



(c) Loading 500 input documents from disk before making any API calls.



(d) Streaming the final compiled report to a user via SSE.


<br>

In [ ]:
# CONCEPT CHECK: genuine asyncio use case
### YOUR CODE HERE ###
answer = "..."  # write 'a', 'b', 'c', or 'd' and one sentence

<hr style="border: 2px solid#003262;" />

<!-- ANSWER (for reviewer): (b) is correct. 500 independent I/O-bound requests benefit from asyncio.gather or a thread pool. (b) is wrong: one blocking sleep per poll is not a concurrency problem. (c) is partially correct (SSE + async server), but it is a streaming concern, not a batch concurrency concern. (d) is wrong: disk I/O at this scale is not an asyncio problem. -->

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<hr style="border: 2px solid#003262;" />

## PART 3: **Error Handling** - Transient vs. Terminal

```mermaid
flowchart TD
    E["an exception from the SDK"] --> Q{"can a later identical<br/>call succeed?"}
    Q -->|"yes - 429, 503, 529"| T["transient:<br/>retry with exponential backoff"]
    Q -->|"no - 401, 403, 400"| P["terminal:<br/>raise now with a clear message"]
    T --> C{"attempt cap reached?"}
    C -->|"no"| T
    C -->|"yes"| P
```

Three classes of API errors:

| Class | Status codes | Action |
|---|---|---|
| Transient | 429, 503, network errors | Retry with exponential backoff + jitter |
| Terminal-user | 400, 401, 403, 404 | Raise immediately - retrying will not help |
| Terminal-server | 500 (specific body) | Do not retry; log and escalate |

The critical case: **401 Unauthorized is NOT retryable**. The key is invalid or revoked. Fail fast and surface the error.

SDK exception classes: `RateLimitError` (429), `InternalServerError` (500+), `AuthenticationError` (401), `BadRequestError` (400).

In [10]:
# Tier 1 - error dispatch: retry vs fail-fast
import time
import random

def call_with_dispatch(client, model_id: str, prompt: str, max_retries: int = 3) -> str:
    for attempt in range(max_retries):
        try:
            resp = client.messages.create(
                model=model_id, max_tokens=64,
                messages=[{"role": "user", "content": prompt}],
            )
            return resp.content[0].text
        except anthropic.RateLimitError:
            if attempt == max_retries - 1:
                raise
            time.sleep((2 ** attempt) + random.uniform(0, 0.1))
        except anthropic.InternalServerError:
            if attempt == max_retries - 1:
                raise
            time.sleep((2 ** attempt) + random.uniform(0, 0.1))
        except anthropic.AuthenticationError:
            raise  # 401: fail-fast
        except anthropic.BadRequestError:
            raise  # 400: fail-fast
    raise RuntimeError("unreachable")

print("call_with_dispatch defined.")

call_with_dispatch defined.


In [11]:
# Tier 2 - with_retry from ccdvf encapsulates the same policy
from ccdvf.api.retry import _RETRYABLE_STATUS

print("Retryable codes:", sorted(_RETRYABLE_STATUS))
print("Non-retryable: 400, 401, 403 (raise immediately)")

if not DRY_RUN:
    raw_client = anthropic.Anthropic(api_key=load_api_key())
    text = with_retry(
        lambda: raw_client.messages.create(
            model=haiku.id, max_tokens=16,
            messages=[{"role": "user", "content": "Say retry-safe."}],
        ).content[0].text
    )
    print("Response:", text)
else:
    print("DRY RUN: skipping live call (Part 3 Tier 2)")

Retryable codes: [429, 500, 502, 503, 504]
Non-retryable: 400, 401, 403 (raise immediately)
Response: retry-safe


In [12]:
# Verified against https://docs.anthropic.com/en/api/messages 2026-09-08
# Tier 3 - error injection test to verify retry counts
from unittest.mock import MagicMock, patch
import anthropic as _anthropic
from types import SimpleNamespace

call_count = [0]

def flaky_create(**kwargs):
    call_count[0] += 1
    if call_count[0] < 3:
        raise _anthropic.InternalServerError(
            "server error", response=MagicMock(status_code=500), body={}
        )
    block = SimpleNamespace(type="text", text="retry-succeeded")
    return SimpleNamespace(
        content=[block], stop_reason="end_turn",
        usage=SimpleNamespace(input_tokens=5, output_tokens=3), model=haiku.id
    )

with patch("ccdvf.api.retry.time.sleep"):  # skip real sleep in tests
    result = with_retry(lambda: flaky_create(model=haiku.id, max_tokens=16, messages=[]))
print(f"Succeeded after {call_count[0]} attempt(s). Response: {result.content[0].text}")

Succeeded after 3 attempt(s). Response: retry-succeeded


<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **Your Messages API call raises `anthropic.APIStatusError` with `status_code == 401`. Should your code retry?**

(a) No - but swallow the error quietly and return an empty response to the caller so the pipeline keeps running.

(b) Yes - retry up to 3 times; a 401 can indicate a transient auth service blip.

(c) Yes - but only once, as a quick confirmation of whether the failure was transient.

(d) No - a 401 means the key is missing, invalid, or revoked. Fail fast and surface the error.

<br>

In [ ]:
# CONCEPT CHECK: retry policy for 401
### YOUR CODE HERE ###
answer = "..."  # write 'a', 'b', 'c', or 'd'

<hr style="border: 2px solid#003262;" />

<!-- ANSWER (for reviewer): (d) is correct. 401 is a terminal error - the key does not change between retries. Retrying wastes latency. Fail fast so the caller can fix the key. (a) and (c) are wrong: auth errors are not transient. (d) is wrong: silent failure is worse than raising. -->

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<hr style="border: 2px solid#003262;" />

## PART 4: **Refactoring** - A Working Script Becomes a Testable Module

Every Claude integration starts as inline code that works. Refactoring earns its cost when the same code will run more than once, be tested, or be maintained by someone else. This Part shows three stages of a document summarizer.

In [14]:
# Verified against https://docs.anthropic.com/en/api/messages 2026-09-08
# Tier 1 - prototype: works, but hard to test or reuse
def summarize_v1(text: str) -> str:
    client = anthropic.Anthropic()  # new client per call
    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",  # hard-coded model ID
        max_tokens=128,
        messages=[{"role": "user", "content": f"Summarize in one sentence: {text}"}],
    )
    return resp.content[0].text

print("Prototype problems:")
print("  1. New client per call (connection overhead)")
print("  2. Hard-coded model ID")
print("  3. No error handling or retry")
print("  4. Not testable without real API")

Prototype problems:
  1. New client per call (connection overhead)
  2. Hard-coded model ID
  3. No error handling or retry
  4. Not testable without real API


In [15]:
# Tier 2 - refactored class: injectable, inspectable, testable
from dataclasses import dataclass, field
from ccdvf.config.model_registry import cost_for_call, get_model

@dataclass
class Summarizer:
    """Summarizes text using Claude.

    Preconditions: client responds to client.messages.create(**kwargs).
    Postconditions: summarize() returns non-empty str; total_cost_usd increases.
    """
    client: object
    model_key: str = "haiku_current"
    max_tokens: int = 128
    total_cost_usd: float = field(default=0.0)
    call_count: int = field(default=0)

    def summarize(self, text: str) -> str:
        spec = get_model(self.model_key)
        resp = self.client.messages.create(
            model=spec.id, max_tokens=self.max_tokens,
            messages=[{"role": "user", "content": f"Summarize in one sentence: {text}"}],
        )
        self.total_cost_usd += cost_for_call(
            self.model_key,
            input_tokens=resp.usage.input_tokens,
            output_tokens=resp.usage.output_tokens,
        )
        self.call_count += 1
        return resp.content[0].text

# Mock the client: no real API needed
mock_resp = SimpleNamespace(
    content=[SimpleNamespace(text="Mock summary.")],
    usage=SimpleNamespace(input_tokens=10, output_tokens=5),
)
mock_client = SimpleNamespace(messages=SimpleNamespace(create=lambda **kw: mock_resp))
s = Summarizer(client=mock_client)
print("Result:", s.summarize("The sky is blue."))
print(f"call_count={s.call_count}, cost=${s.total_cost_usd:.6f}")

Result: Mock summary.
call_count=1, cost=$0.000035


In [16]:
# Tier 3 - production shape from ccdvf with retry, budget, frozen CallResult
if not DRY_RUN:
    with MessagesClient(model="haiku_current", budget_usd=0.02) as mc:
        result = mc.call(
            messages=[{"role": "user", "content": "Summarize in one sentence: The sky is blue."}],
            max_tokens=64,
        )
    print(f"text: {result.text!r}")
    print(f"Spent: ${mc.budget.spent_usd:.6f} over {mc.budget.call_count} call(s)")
else:
    import inspect
    print("MessagesClient contract summary:")
    print(inspect.getdoc(MessagesClient)[:300])

text: 'The sky appears blue in color.'
Spent: $0.000069 over 1 call(s)


<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **The tier 2 refactor changed the summarization script to a class. Name two testability improvements that came with the class shape.**

(a) The class runs noticeably faster, since Python only allocates the client object once per run.

(b) The `@dataclass` decorator auto-generates `__eq__`, which makes writing equality assertions simpler.

(c) The class automatically retries on 429 errors, resolved through Python's method resolution order.

(d) The client can be injected and mocked, and per-instance state is inspectable in assertions.

<br>

In [ ]:
# CONCEPT CHECK: testability from the class refactor
### YOUR CODE HERE ###
answer = "..."  # write 'a', 'b', 'c', or 'd'

<hr style="border: 2px solid#003262;" />

<!-- ANSWER (for reviewer): (d) is correct. Dependency injection enables mocking; per-instance state enables assertions on spent/count. (a) is wrong: speed is not testability. (b) is partially true but not the primary testability gain. (c) is wrong: retry requires explicit logic. -->

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<hr style="border: 2px solid#003262;" />

## PART 4b: **Version Control** for Claude API Code

Three git-relevant surfaces unique to Claude API codebases:

1. **Prompts are code** - store in versioned `.txt` files, not inline strings, so `git blame` identifies when a prompt changed. See `ccdvf/prompts` and `ccdvf/config` for the full implementation.

2. **Model registry entries are code** - a `git diff` on `model_registry.py` must be reviewable like any other API-surface change.

3. **Notebooks need `nbstripout`** - raw `.ipynb` files contain outputs and execution counts that pollute diffs. A pre-commit hook with `nbstripout` ensures only source-cell changes surface.

In [17]:
# Tier 1 - show git log on a versioned Python module
result_p = subprocess.run(
    ["git", "log", "--oneline", "-5", "ccdvf/config/model_registry.py"],
    capture_output=True, text=True, check=False
)
if result_p.returncode == 0 and result_p.stdout.strip():
    print("git log on model_registry.py:")
    print(result_p.stdout)
else:
    print("(git history grows as commits accumulate)")

git log on model_registry.py:
b304189 Initial commit



In [18]:
# Tier 2 - show the nbstripout pre-commit config snippet
snippet = (
    "repos:\n"
    "  - repo: https://github.com/kynan/nbstripout\n"
    "    rev: 0.7.1\n"
    "    hooks:\n"
    "      - id: nbstripout\n"
)
print("Pre-commit config for notebook diff hygiene:")
print(snippet)
print("Effect: git diff shows only source cell changes, not output blobs.")

Pre-commit config for notebook diff hygiene:
repos:
  - repo: https://github.com/kynan/nbstripout
    rev: 0.7.1
    hooks:
      - id: nbstripout

Effect: git diff shows only source cell changes, not output blobs.


In [19]:
# Tier 3 - check if nbstripout is available in this environment
for tool in ["nbstripout", "jupytext"]:
    r = subprocess.run([sys.executable, "-m", "pip", "show", tool], capture_output=True, text=True, check=False)
    status = "installed" if r.returncode == 0 else "not installed"
    print(f"{tool}: {status}")

nbstripout: not installed
jupytext: not installed


<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **A team's Claude API quality dropped 12% after a "small refactor" commit. `git log` shows a one-character change in a prompt file. Which remediation is most appropriate?**

Rollback, then: 

(a) Keep prompts in versioned files, and add a CI evaluation step that gates quality before any prompt change merges.

(b) Move prompts into a database table, so every change requires a formal schema migration to deploy.

(c) Keep prompts inline in application code, where they are supposedly harder to change by accident.

(d) Add the prompt file to `.gitignore`, so future changes require explicit manual approval before they take effect.

<br>

In [ ]:
# CONCEPT CHECK: prompt regression remediation
### YOUR CODE HERE ###
answer = "..."  # write 'a', 'b', 'c', or 'd'

<hr style="border: 2px solid#003262;" />

<!-- ANSWER (for reviewer): (a) is correct. The problem was not that the file was versioned - it was that there was no quality gate. Add CI eval on prompt changes. (a) is wrong: inline prompts are harder to track, not easier. (b) is wrong: gitignoring removes auditability entirely. (d) is wrong: a database adds operational cost without improving auditability. -->

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<hr style="border: 2px solid#003262;" />

## PART 4c: **Pytest** Basics for Testing Claude Integrations

The Tier 2 refactor in Part 4 made `Summarizer` testable by injecting the client. This Part writes the test that injection enables: a pytest function that never calls `anthropic.Anthropic()` for real.

Three conventions pytest expects:

- Test files and functions start with `test_` so pytest discovers them automatically.
- Each test follows **Arrange - Act - Assert**: build the inputs, call the code, check the result with a plain `assert`.
- External dependencies get replaced with `unittest.mock.patch` so the test is fast, free, and deterministic - the same "stunt double" idea the full suite in `irl_examples/02_software_engineering_foundations/integration_mocking/` uses for `CodeReviewTriager`.

In [ ]:
# Tier 1 - anatomy of a pytest test: Arrange, Act, Assert
import subprocess, sys, tempfile, textwrap
from pathlib import Path

with tempfile.TemporaryDirectory() as tier1_dir_str:
    tier1_dir = Path(tier1_dir_str)
    (tier1_dir / "calc.py").write_text(textwrap.dedent("""
        def cost_usd(input_tokens: int, output_tokens: int, input_rate: float, output_rate: float) -> float:
            return input_tokens * input_rate + output_tokens * output_rate
"""))
    (tier1_dir / "test_calc.py").write_text(textwrap.dedent("""
        from calc import cost_usd

        def test_cost_usd_combines_input_and_output():
            # Arrange
            input_tokens, output_tokens = 500, 10
            # Act
            cost = cost_usd(input_tokens, output_tokens, input_rate=1.00/1_000_000, output_rate=5.00/1_000_000)
            # Assert
            assert round(cost, 6) == round(500 * (1.00/1_000_000) + 10 * (5.00/1_000_000), 6)
"""))

    result = subprocess.run(
        [sys.executable, "-m", "pytest", "-v", str(tier1_dir)],
        capture_output=True, text=True, cwd=tier1_dir,
    )
    # returncode 0/1 means pytest ran and reported results (pass/fail); anything
    # else (missing pytest, collection error) means pytest itself didn't finish.
    if result.returncode not in (0, 1):
        print("pytest did not run cleanly - check that this notebook's kernel has pytest installed:")
        print((result.stdout.strip() or result.stderr)[-600:])
    else:
        print(result.stdout[-600:])

In [ ]:
# Tier 2 - mocking anthropic.Anthropic so no real API call happens
with tempfile.TemporaryDirectory() as tier2_dir_str:
    tier2_dir = Path(tier2_dir_str)
    (tier2_dir / "triager.py").write_text(textwrap.dedent("""
        import anthropic

        class BudgetExceededError(Exception):
            pass

        class TicketTriager:
            def __init__(self, model_key: str, budget_usd: float):
                self.model_key = model_key
                self.budget_usd = budget_usd
                self.spent_usd = 0.0
                self.call_count = 0
                self.client = anthropic.Anthropic()

            def triage(self, ticket: dict) -> str:
                if self.spent_usd >= self.budget_usd or self.budget_usd <= 0:
                    raise BudgetExceededError("Operational budget limit hit.")
                resp = self.client.messages.create(
                    model=self.model_key, max_tokens=10,
                    messages=[{"role": "user", "content": ticket["title"]}],
                )
                self.spent_usd += 0.0001
                self.call_count += 1
                return resp.content[0].text.strip().lower()
"""))
    (tier2_dir / "test_triager.py").write_text(textwrap.dedent('''
        from unittest.mock import MagicMock, patch
        import pytest
        from triager import TicketTriager, BudgetExceededError

        @pytest.fixture
        def mock_anthropic():
            with patch("triager.anthropic.Anthropic") as mock_class:
                mock_instance = MagicMock()
                mock_class.return_value = mock_instance
                yield mock_instance

        def test_triage_returns_label_without_a_real_api_call(mock_anthropic):
            mock_anthropic.messages.create.return_value = MagicMock(
                content=[MagicMock(text="billing")]
            )
            triager = TicketTriager(model_key="claude-haiku-4-5-20251001", budget_usd=1.0)
            assert triager.triage({"title": "Charged twice"}) == "billing"
            mock_anthropic.messages.create.assert_called_once()

        def test_budget_guardrail_blocks_the_call_before_it_happens(mock_anthropic):
            triager = TicketTriager(model_key="claude-haiku-4-5-20251001", budget_usd=0.0)
            with pytest.raises(BudgetExceededError):
                triager.triage({"title": "Should never reach the client"})
            mock_anthropic.messages.create.assert_not_called()
'''))

    result = subprocess.run(
        [sys.executable, "-m", "pytest", "-v", str(tier2_dir)],
        capture_output=True, text=True, cwd=tier2_dir,
    )
    # returncode 0/1 means pytest ran and reported results (pass/fail); anything
    # else (missing pytest/anthropic, collection error) means pytest itself didn't finish.
    if result.returncode not in (0, 1):
        print("pytest did not run cleanly - check that this notebook's kernel has pytest and anthropic installed:")
        print((result.stdout.strip() or result.stderr)[-800:])
    else:
        print(result.stdout[-800:])

In [ ]:
# Tier 3 - the full offline/online suite lives outside the notebook
print("irl_examples/02_software_engineering_foundations/integration_mocking/ extends this pattern for CodeReviewTriager:")
print("  - conftest.py adds a --online flag so the same tests can hit the live API on demand")
print("  - test_triager.py adds retry, fail-fast, and fallback assertions for each error class from Part 3")
print()
print("Run it from the repo root:")
print("  pytest irl_examples/02_software_engineering_foundations/integration_mocking/ -v          # offline, free, default")
print("  pytest irl_examples/02_software_engineering_foundations/integration_mocking/ -v --online # billable, hits the real API")

<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **Your CI pipeline runs the pytest suite in Part 4c on every pull request, dozens of times a day. Why does Tier 2 patch `anthropic.Anthropic` instead of calling the real API?**

(a) Deterministic, free, offline assertions - CI shouldn't burn tokens or depend on live API uptime for your own logic.

(b) The real API requires a paid Anthropic plan, and pytest has no mechanism to authenticate against it on CI runners.

(c) `unittest.mock.patch` is required by pytest syntax whenever the word `client` appears anywhere in a test.

(d) Mocking makes the test execute the exact same code path as a live call, so the results end up identical either way.

<br>

In [ ]:
# CONCEPT CHECK: why mock the Anthropic client in tests
### YOUR CODE HERE ###
answer = "..."  # write 'a', 'b', 'c', or 'd'

<hr style="border: 2px solid#003262;" />

<!-- ANSWER (for reviewer): (a) is correct. Mocking isolates your code's logic from network latency, API uptime, and token cost - the point is to verify TicketTriager's budget and parsing logic, not Anthropic's servers. (b) is wrong: pytest has no authentication concept. (c) is wrong: patching is a choice, not a syntax requirement. (d) is wrong: a mock runs a stand-in object, not the real call path, which is exactly why it is fast and deterministic. -->

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

<hr style="border: 2px solid#003262;" />

## PART 5: **Code Review** Checklist for Claude Integrations

The 10 most common review comments on a first Claude integration:

| # | Anti-pattern | Correct form |
|---|---|---|
| 1 | Model ID hard-coded at call site | Move to `model_registry.py` |
| 2 | API key written into source | Load from env with `load_api_key()` |
| 3 | No retry on 429 | Add `with_retry` or use `MessagesClient` |
| 4 | Retry on 401 | Remove; 401 is terminal |
| 5 | No `with` block for client | Use context manager to close connection |
| 6 | Tool schema as inline dict | Move to JSON file under `tools/schemas/` |
| 7 | No cost cap | Add `BudgetTracker` or `MessagesClient(budget_usd=...)` |
| 8 | Prompt duplicated across files | Extract to `ccdvf.prompts` module |
| 9 | `except Exception: pass` | Name the exception; log and handle explicitly |
| 10 | Tests hit real API | Mock at `anthropic.Anthropic` boundary |


In [20]:
# Verified against https://docs.anthropic.com/en/api/messages 2026-09-08
# Tier 1 - display a problematic code snippet for review
bad_snippet = (
    "try:\n"
    "    response = client.messages.create(\n"
    "        model='claude-haiku-4-5-20251001',\n"
    "        max_tokens=512,\n"
    "        messages=[{'role': 'user', 'content': prompt}],\n"
    "    )\n"
    "except Exception:\n"
    "    response = None\n"
)
print("Code under review:")
print(bad_snippet)
print("Name at least two specific problems and their remediations.")

Code under review:
try:
    response = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=512,
        messages=[{'role': 'user', 'content': prompt}],
    )
except Exception:
    response = None

Name at least two specific problems and their remediations.


<!--Navigate back to table of contents-->
<div align="left" style="text-align: left; background-color:#003262;">
    <span>
        <hr style="border: 8px solid#003262;" />
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:0px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:24px;letter-spacing:0px;line-height:20px;padding:24px 40px;text-align:left;text-decoration:none; align:left"> 
            <strong>CONCEPT</strong> CHECK 
        </a>        
    </span>
</div>
<!-------------------------------------->

> **Given the snippet above (bare `except Exception: response = None`), name at least two specific problems and their remediations.**

(a) The `client` variable should be passed not hardcoded.

(b) Bare `except Exception` swallows every error including keyboard interrupt and terminal errors like 401; catch specific `anthropic.APIStatusError` subclasses.

(c) The `messages` key should be a string, not a list.

(d) The `max_tokens` of 512 is too high for production.

(e) On failure, `response = None` propagates a broken state with no signal; raise or use a typed error result

<br>

In [ ]:
# CONCEPT CHECK: code review - two problems in the bare except snippet
### YOUR CODE HERE ###
answer = "..."  # write 'a', 'b', 'c', or 'd'

<hr style="border: 2px solid#003262;" />

<!-- ANSWER (for reviewer): (a),(b),(d) are correct: (1) hard-coding variables that should be passed make your code brittle. (2) bare except swallows everything (fix: catch anthropic.APIStatusError subclasses), (3) response = None on failure propagates a broken state (fix: raise or use a typed error result). A third unlisted problem: no logging means the failure vanishes with no observability. (a) is wrong: client is in scope. (c) is wrong: messages must be a list - that is correct. (d) is wrong: 512 is fine. -->

<!--Navigate back to table of contents-->
<div alig="right" style="text-align: right">
    <span>
        <a style="color:#FFFFFF; background-color:#003262; border:1px solid #FFFFFF; border-color:#FFFFFF;border-radius:5px;border-width:0px;display:inline-block;font-family:arial,helvetica,sans-serif;font-size:10px;letter-spacing:0px;line-height:10px;padding:10px 20px;text-align:center;text-decoration:none; align:center" href="#Part_table_contents" name="Table of Contents"  id="Part_table_contents"> 
            Table of Contents 
        </a>
    </span>
</div>
<!-------------------------------------->

## Sources Consulted

- [Anthropic API Getting Started](https://platform.claude.com/docs/en/api/getting-started) - retrieved 2026-09-02; volatile
- [Messages API Reference](https://platform.claude.com/docs/en/api/messages) - retrieved 2026-09-02; volatile
- [Anthropic Python SDK](https://github.com/anthropics/anthropic-sdk-python) - retrieved 2026-09-02; volatile

<hr style="border: 6px solid#003262;" />